In [7]:
import re
import math
import pandas as pd
from urllib.parse import urlparse
from collections import Counter

# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------

SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "secure", "account", "update",
    "banking", "confirm", "password", "paypal", "ebay", "amazon",
    "apple", "microsoft", "wallet", "free", "prize", "click", "alert",
]

IP_PATTERN = re.compile(
    r"(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)"
)

SCHEMES = ("https://", "http://", "ftp://", "ftps://")


def normalize_url(url: str) -> str:
    """Strip scheme and ensure a path exists, so features match what a
    browser will feed the extension at inference time (browsers always
    expose at least '/' as pathname)."""
    url = str(url).strip()
    lower = url.lower()
    for s in SCHEMES:
        if lower.startswith(s):
            url = url[len(s):]
            break
    # Bare domain -> append '/' so path_length and path_depth match
    # the distribution the extension will produce in production.
    if "/" not in url:
        url = url + "/"
    return url


def entropy(s: str) -> float:
    if not s:
        return 0.0
    counts = Counter(s)
    total = len(s)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def extract_features(url: str) -> dict:
    # Normalize FIRST — then prepend a dummy scheme so urlparse works
    url = normalize_url(url)
    try:
        parsed = urlparse("http://" + url)
    except Exception:
        parsed = urlparse("http://")

    netloc   = parsed.netloc or ""
    path     = parsed.path or ""
    query    = parsed.query or ""

    domain = netloc.split(":")[0]
    tld    = domain.rsplit(".", 1)[-1] if "." in domain else ""
    full   = url.lower()

    f = {}
    # --- Length ---
    f["url_length"]     = len(url)
    f["domain_length"]  = len(domain)
    f["path_length"]    = len(path)
    f["query_length"]   = len(query)

    # --- Counts ---
    f["dot_count"]        = url.count(".")
    f["hyphen_count"]     = url.count("-")
    f["underscore_count"] = url.count("_")
    f["slash_count"]      = url.count("/")
    f["at_count"]         = url.count("@")
    f["question_count"]   = url.count("?")
    f["amp_count"]        = url.count("&")
    f["eq_count"]         = url.count("=")
    f["percent_count"]    = url.count("%")
    f["hash_count"]       = url.count("#")
    f["digit_count"]      = sum(c.isdigit() for c in url)
    f["letter_count"]     = sum(c.isalpha() for c in url)
    f["digit_ratio"]      = f["digit_count"] / max(len(url), 1)

    # --- Structure ---
    parts = domain.split(".")
    f["subdomain_count"]   = max(len(parts) - 2, 0)
    f["domain_entropy"]    = entropy(domain)
    f["path_entropy"]      = entropy(path)
    f["path_depth"]        = path.count("/")
    f["query_param_count"] = len(query.split("&")) if query else 0
    f["tld_length"]        = len(tld)

    # --- Flags ---
    # NOTE: is_https removed — leaked in training data
    f["has_ip"]           = int(bool(IP_PATTERN.search(netloc)))
    f["has_port"]         = int(":" in netloc)
    f["has_at"]           = int("@" in url)
    f["has_double_slash"] = int("//" in path)
    f["has_hex_chars"]    = int(bool(re.search(r"%[0-9a-fA-F]{2}", url)))

    # --- Semantic ---
    f["suspicious_keyword_count"] = sum(kw in full for kw in SUSPICIOUS_KEYWORDS)

    shorteners = ["bit.ly", "tinyurl", "goo.gl", "t.co", "ow.ly",
                  "is.gd", "buff.ly", "adf.ly", "shorte.st"]
    f["is_shortened"] = int(any(s in full for s in shorteners))

    risky_tlds = {"tk", "ml", "ga", "cf", "gq", "xyz", "top",
                  "club", "work", "date", "racing", "review"}
    f["risky_tld"] = int(tld.lower() in risky_tlds)

    return f


# ---------------------------------------------------------------------------
# Label encoding
# ---------------------------------------------------------------------------

LABEL_MAP = {
    "benign":      0,
    "defacement":  1,
    "phishing":    2,
    "malware":     3,
}


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def build_dataset(input_csv: str, output_csv: str):
    print(f"Loading {input_csv} ...")
    df = pd.read_csv(input_csv)

    print("Extracting features ...")
    features = df["url"].apply(extract_features).apply(pd.Series)
    features["label"] = df["type"].map(LABEL_MAP)

    print(f"Feature matrix shape: {features.shape}")
    print(features.head())

    features.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    return features


if __name__ == "__main__":
    build_dataset(
        input_csv="malicious_phish.csv",
        output_csv="url_features.csv",
    )

Loading malicious_phish.csv ...
Extracting features ...
Feature matrix shape: (651191, 32)
   url_length  domain_length  path_length  query_length  dot_count  \
0        17.0           16.0          1.0           0.0        2.0   
1        35.0           11.0         24.0           0.0        2.0   
2        31.0           14.0         17.0           0.0        2.0   
3        81.0           21.0         10.0          49.0        3.0   
4       228.0           23.0         10.0         194.0        2.0   

   hyphen_count  underscore_count  slash_count  at_count  question_count  ...  \
0           1.0               0.0          1.0       0.0             0.0  ...   
1           0.0               1.0          2.0       0.0             0.0  ...   
2           0.0               0.0          3.0       0.0             0.0  ...   
3           1.0               2.0          1.0       0.0             1.0  ...   
4           1.0               1.0          1.0       0.0             1.0  ...   

 

In [3]:
!pip install \
pandas>=2.0 \
scikit-learn>=1.3 \
lightgbm>=4.0 \
fastapi>=0.110 \
uvicorn[standard]>=0.27 \
pydantic>=2.0

In [8]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

FEATURE_CSV  = "url_features.csv"
MODEL_OUT    = "url_model.txt"
METRICS_OUT  = "metrics.json"
FEATURES_OUT = "feature_names.json"

LABEL_NAMES = ["benign", "defacement", "phishing", "malware"]

PARAMS = {
    "objective":        "multiclass",
    "num_class":        4,
    "metric":           "multi_logloss",
    "learning_rate":    0.05,
    "num_leaves":       63,
    "max_depth":        -1,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "verbose":          -1,
    "n_jobs":           -1,
}

NUM_BOOST_ROUND = 500
EARLY_STOPPING  = 30


def main():
    print(f"Loading {FEATURE_CSV} ...")
    df = pd.read_csv(FEATURE_CSV)

    X = df.drop(columns=["label"])
    y = df["label"]
    feature_names = X.columns.tolist()

    print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} features")
    print(f"Class distribution:\n{y.value_counts().sort_index()}\n")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # ---- Class balancing via per-sample weights ----
    classes = np.array(sorted(y_train.unique()))
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    class_to_weight = dict(zip(classes, weights))
    sample_weight = y_train.map(class_to_weight).values

    print("Per-class weights:")
    for c, w in class_to_weight.items():
        print(f"  {LABEL_NAMES[c]:12s} {w:.4f}")

    train_set = lgb.Dataset(X_train, label=y_train, weight=sample_weight)
    valid_set = lgb.Dataset(X_test,  label=y_test, reference=train_set)

    print("\nTraining LightGBM ...")
    model = lgb.train(
        PARAMS,
        train_set,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[valid_set],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING),
            lgb.log_evaluation(50),
        ],
    )

    # ---- Evaluate ----
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)
    y_pred_labels = y_pred.argmax(axis=1)

    acc = accuracy_score(y_test, y_pred_labels)
    report = classification_report(
        y_test, y_pred_labels, target_names=LABEL_NAMES, output_dict=True
    )
    cm = confusion_matrix(y_test, y_pred_labels).tolist()

    print(f"\nAccuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred_labels, target_names=LABEL_NAMES))
    print("Confusion matrix (rows=true, cols=pred):")
    for row in cm:
        print(row)

    importance = sorted(
        zip(feature_names, model.feature_importance(importance_type="gain")),
        key=lambda x: -x[1],
    )
    print("\nTop 10 features by gain:")
    for name, score in importance[:10]:
        print(f"  {name:28s} {score:.1f}")

    # ---- Save ----
    model.save_model(MODEL_OUT, num_iteration=model.best_iteration)
    print(f"\nModel saved: {MODEL_OUT}")

    with open(FEATURES_OUT, "w") as f:
        json.dump(feature_names, f)
    print(f"Feature names saved: {FEATURES_OUT}")

    with open(METRICS_OUT, "w") as f:
        json.dump({
            "accuracy":              acc,
            "best_iteration":        model.best_iteration,
            "classification_report": report,
            "confusion_matrix":      cm,
            "feature_importance":    [[n, float(s)] for n, s in importance],
        }, f, indent=2)
    print(f"Metrics saved: {METRICS_OUT}")


if __name__ == "__main__":
    main()

Loading url_features.csv ...
Dataset: 651191 rows, 31 features
Class distribution:
label
0    428103
1     96457
2     94111
3     32520
Name: count, dtype: int64

Per-class weights:
  benign       0.3803
  defacement   1.6878
  phishing     1.7298
  malware      5.0061

Training LightGBM ...
Training until validation scores don't improve for 30 rounds
[50]	valid's multi_logloss: 0.573332
[100]	valid's multi_logloss: 0.486021
[150]	valid's multi_logloss: 0.451418
[200]	valid's multi_logloss: 0.430961
[250]	valid's multi_logloss: 0.415521
[300]	valid's multi_logloss: 0.40274
[350]	valid's multi_logloss: 0.393221
[400]	valid's multi_logloss: 0.385461
[450]	valid's multi_logloss: 0.377185
[500]	valid's multi_logloss: 0.371502
Did not meet early stopping. Best iteration is:
[500]	valid's multi_logloss: 0.371502

Accuracy: 0.8343
              precision    recall  f1-score   support

      benign       0.96      0.80      0.87     85621
  defacement       0.85      0.94      0.89     19292


In [10]:
import json
import pandas as pd
import lightgbm as lgb

MODEL_PATH    = "url_model.txt"
FEATURES_PATH = "feature_names.json"
LABEL_NAMES   = ["benign", "defacement", "phishing", "malware"]

class URLClassifier:
    def __init__(self, model_path=MODEL_PATH, features_path=FEATURES_PATH):
        self.model = lgb.Booster(model_file=model_path)
        with open(features_path) as f:
            self.feature_names = json.load(f)

    def predict(self, url: str) -> dict:
        # Ensure extract_features is defined earlier in the notebook
        feats = extract_features(url)
        X = pd.DataFrame([feats])[self.feature_names]
        probs = self.model.predict(X)[0]
        label_idx = int(probs.argmax())
        return {
            "url":        url,
            "label":      LABEL_NAMES[label_idx],
            "confidence": float(probs[label_idx]),
            "scores":     {n: float(p) for n, p in zip(LABEL_NAMES, probs)},
        }

if __name__ == "__main__":
    clf = URLClassifier()

    test_urls = [
        "https://br-icloud.com.br",
        "http://192.168.1.1/login.php?user=admin",
        "http://paypa1-verify-account.tk/login",
        "https://github.com/anthropics/claude-code",
        "http://bit.ly/3xR4nd0m",
        "http://secure-update-apple.xyz/verify?id=0x8f4a",
    ]

    for url in test_urls:
        result = clf.predict(url)
        print(f"{result['label']:12s} ({result['confidence']:.2%})  {url}")


phishing     (75.84%)  https://br-icloud.com.br
phishing     (88.45%)  http://192.168.1.1/login.php?user=admin
phishing     (83.88%)  http://paypa1-verify-account.tk/login
benign       (88.34%)  https://github.com/anthropics/claude-code
phishing     (88.47%)  http://bit.ly/3xR4nd0m
phishing     (98.73%)  http://secure-update-apple.xyz/verify?id=0x8f4a
